<!--
Copyright (c) 2026 OceanBase.

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.
-->

# 13 · 开发者提交，审核者批准

团队协作需要明确谁能提交材料、谁能批准候选、谁只能查看。我们会用三个不同身份发送真实 HTTP 请求，然后撤销一个授权，检查后续请求和审计记录。

无需模型。这里使用正式 Authentication Provider 扩展接口接入一组仅在本次实验有效的随机凭证，Authorization 使用产品的内置数据库实现。这个小身份提供器用于教学，并不是账号注册或生产登录系统。凭证不会显示在输出中。

路线：准备身份 → 分配 Scope 角色 → 提交候选 → 拒绝越权审批 → 审核者批准 → 精确只读分享 → 撤销与审计。

In [ ]:
import sys
from pathlib import Path

from _tutorial import Tutorial, show

from powercontext.http import CreateScopeRequest

if not Path("_tutorial.py").is_file():
    sys.path.insert(0, str(Path.cwd() / "examples" / "jupyter"))
if previous_lab := globals().get("lab"):
    await previous_lab.close()
lab = await Tutorial.start("13", features=())
client = lab.client
assert client is not None
scope = await client.create_scope(
    CreateScopeRequest(
        title="订单 CSV 导入器 · 13", summary="本次教学实验的独立材料", idempotency_key=f"{lab.run_id}:main"
    )
)
scope_id = scope.scope_id

## 给真实请求附上不同身份

先停止服务，再在同一实验数据库中初始化管理员。重新启动时，每个随机 Token 都由服务端映射到固定 Principal。请求 JSON 里填写用户名，不能改变调用者身份。

In [ ]:
import secrets
from secrets import compare_digest

from powercontext.client import ForbiddenResponseError, PowerContextClient
from powercontext.server.authentication import AuthenticationRejectedError, AuthenticationResult, ProviderReadiness
from powercontext.server.authz import PrincipalRef
from powercontext.server.authz.composition import open_builtin_access_control
from powercontext.server.settings import AccessControlConfig, BearerAuthConfig

principals = {
    name: PrincipalRef(type="user", id=f"{lab.run_id}-{name}") for name in ("admin", "developer", "reviewer", "viewer")
}
tokens = {name: secrets.token_urlsafe(32) for name in principals}


class TutorialIdentities:
    async def authenticate(self, request, /):
        supplied = request.headers.get("authorization", "").removeprefix("Bearer ")
        for name, token in tokens.items():
            if compare_digest(supplied, token):
                return AuthenticationResult(subject=principals[name])
        raise AuthenticationRejectedError

    async def readiness(self):
        return ProviderReadiness(ready=True)


await lab.close()
async with open_builtin_access_control(
    lab.database, bootstrap_administrators=(principals["admin"],), deployment_id=lab.run_id
):
    pass
lab.settings_overrides.update(
    access=AccessControlConfig(mode="enforced", deployment_id=lab.run_id), auth=BearerAuthConfig(enabled=False)
)
lab.app_options["authentication_provider"] = TutorialIdentities()
lab.client_token = tokens["admin"]
await lab.restart()
client = lab.client
# Create the Scope in enforced mode to establish ownership.
scope = await client.create_scope(
    CreateScopeRequest(title="团队审核实验", summary="四个独立身份", idempotency_key=f"{lab.run_id}:secured")
)
scope_id = scope.scope_id
actors = {name: PowerContextClient(lab.base_url, token=tokens[name]) for name in ("developer", "reviewer", "viewer")}
show({"身份数": len(principals), "访问模式": (await client.get_access_principal()).mode})

## 授予提交与审核能力

我们直接使用公开角色和绑定 API。开发者和审核者分别获得 Scope contributor、reviewer；只读成员稍后仅获得一个制品的读取权。

In [ ]:
from powercontext.http import (
    CreateAccessBindingRequest,
    CreateSourceRequest,
    ExperienceProposal,
    ProposeExperienceRequest,
    SourceReference,
)

resource = {"type": "scope", "scope_id": scope_id}
for name, role in (("developer", "scope.contributor"), ("reviewer", "scope.reviewer")):
    await client.create_access_binding(
        CreateAccessBindingRequest.model_validate({
            "subject": {"type": "user", "id": principals[name].id},
            "resource": resource,
            "role": role,
            "idempotency_key": f"{lab.run_id}:{name}",
        })
    )
source = await actors["developer"].create_source(
    scope_id, CreateSourceRequest(content="教学输入：Decimal 转分之前检查精度；1.999 应拒绝。")
)
candidate = await actors["developer"].propose_experience(
    ProposeExperienceRequest(
        scope_id=scope_id,
        proposal=ExperienceProposal(
            situation="amount 有超精度输入",
            action="先校验精度",
            outcome="本例指定拒绝 1.999",
            lesson="amount: 校验精度后再转分",
        ),
        source_refs=[SourceReference(name="content", source_id=source.source_id)],
        artifact_refs=[],
    )
)
assert candidate.status == "pending"
show({"提交者": "developer", "候选状态": candidate.status})

## 同一个批准请求，由不同身份执行

先让开发者尝试批准，必须收到 Forbidden。随后使用审核者批准。权限由服务端在操作时重新检查，前端隐藏按钮不能替代这一步。

In [ ]:
from powercontext.http import ApproveArtifactCandidateRequest

approval = ApproveArtifactCandidateRequest(
    scope_id=scope_id, candidate_id=candidate.candidate_id, expected_version=candidate.version
)
try:
    await actors["developer"].approve_artifact_candidate(approval)
except ForbiddenResponseError:
    print("开发者自行批准：被服务端拒绝。")
else:
    raise AssertionError("预期越权请求被拒绝")
approved = await actors["reviewer"].approve_artifact_candidate(approval)
assert approved.status == "approved" and approved.result_artifact
ref = approved.result_artifact
print("审核者批准：已形成制品版本", ref.revision)

## 只分享一个制品，然后撤销

只读成员先无法读取；获得 artifact.viewer 后能读取这一个制品；撤销后再次拒绝。读者可以比较“获得登录凭证”和“拥有资源权限”的区别。

In [ ]:
from powercontext.http import ListAccessAuditRequest, RevokeAccessBindingRequest


async def viewer_read():
    return await actors["viewer"].get_artifact(scope_id, ref.family, ref.artifact_id)


try:
    await viewer_read()
except ForbiddenResponseError:
    pass
else:
    raise AssertionError("未授权成员不应读取制品")
artifact_resource = {
    "type": "artifact",
    "scope_id": scope_id,
    "identity": {"family": ref.family, "artifact_id": ref.artifact_id},
    "selector": None,
}
grant = await client.create_access_binding(
    CreateAccessBindingRequest.model_validate({
        "subject": {"type": "user", "id": principals["viewer"].id},
        "resource": artifact_resource,
        "role": "artifact.viewer",
        "idempotency_key": f"{lab.run_id}:read-one",
    })
)
assert (await viewer_read()).artifact_id == ref.artifact_id
revoked = await client.revoke_access_binding(
    RevokeAccessBindingRequest(
        binding_id=grant.binding_id, expected_version=grant.version, idempotency_key=f"{lab.run_id}:revoke"
    )
)
assert revoked.state == "revoked"
try:
    await viewer_read()
except ForbiddenResponseError:
    print("撤销后新请求：拒绝读取。")
else:
    raise AssertionError("撤销应即时影响后续读取")
audit = await client.list_access_audit(ListAccessAuditRequest.model_validate({"resource": resource, "limit": 100}))
assert audit.items
show(audit)
for actor in actors.values():
    await actor.aclose()

## 练习与验收

用 viewer 尝试创建 Source，预期仍被拒绝。不要把读取某个制品的授权当作整个项目的贡献权。退出前确认审计记录中有实际操作与授权决定。

接下来阅读 [14_continuous_source_ingestion.ipynb](14_continuous_source_ingestion.ipynb)。

最后关闭服务。实验文件保留在本次 `.powercontext/` 目录，便于复查。

In [ ]:
await lab.close()
print("本篇 Server 已关闭。")